### Import Libraries and Reproducability

In [ ]:
import sys, os, re, json, pandas as pd, numpy as np
from datetime import datetime
from collections import Counter
from tqdm.auto import tqdm
import transformers

SEED = 22
transformers.set_seed(SEED)
rng  = np.random.default_rng(SEED)

### Configuration and Instructions

In [ ]:
TRAIN_PATH = "train_w1.csv"
TEST_PATH  = "test_w1.csv"
LABEL      = "SNAP"
OUTPUT_DIR = "./results_wave1"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_ID = "mlx-community/Qwen2.5-1.5B-Instruct-4bit"

DEMO_FEATURES = [
    "marital.status", "sex", "hispanic.origin", "race",
    "education", "citizenship", "employment", "state", "age",
    "income"
]
FULL_FEATURES    = DEMO_FEATURES

COL_MAP = {
    "marital.status":  "Marital Status",
    "sex":             "Sex",
    "hispanic.origin": "Hispanic Origin",
    "race":            "Race",
    "education":       "Educational Attainment",
    "citizenship":     "Citizenship",
    "employment":      "Employment Status",
    "state":           "State of Residence",
    "age":             "Age",
    "income":          "Income",
    "SNAP":            "Coverage for SNAP in Wave 1",
}

INSTRUCTION_STRUCTURED = (
    "You are an expert text classifier. "
    "The Supplemental Nutrition Assistance Program (SNAP) provides food-purchasing "
    "assistance to low-income individuals and families in the U.S. "
    "You will be given a respondent's demographic information and income from "
    "Wave 1 of the 2014 Survey of Income and Program Participation (SIPP). "
    "Based on this information, classify if the person has SNAP coverage in Wave 1. "
    "Return only one label: 'Yes' or 'No', without any other text.\n"
)

INSTRUCTION_NARRATIVE = (
    "You are an expert text classifier. "
    "The Supplemental Nutrition Assistance Program (SNAP) provides food assistance "
    "to low-income households in the U.S. "
    "You will read a short profile of a survey respondent from the 2014 Survey of "
    "Income and Program Participation (SIPP), Wave 1. "
    "Based on the profile, decide whether this person has SNAP coverage. "
    "Return only one label: 'Yes' or 'No', without any other text.\n"
)

### Data Loading

In [ ]:
df_train = pd.read_csv(TRAIN_PATH)
df_test  = pd.read_csv(TEST_PATH)

print(f"Train : {len(df_train):,} rows | SNAP distribution: {dict(Counter(df_train[LABEL]))}")
print(f"Test  : {len(df_test):,}  rows | SNAP distribution: {dict(Counter(df_test[LABEL]))}")

### Oversampling - random oversampling of SNAP = YES

In [ ]:
def oversample_minority(df, label_col, seed=SEED):
    majority = df[df[label_col] == "No"]
    minority = df[df[label_col] == "Yes"]
    minority_upsampled = minority.sample(
        n=len(majority), replace=True, random_state=seed
    )
    return pd.concat([majority, minority_upsampled]).sample(
        frac=1, random_state=seed
    ).reset_index(drop=True)

df_train_balanced = oversample_minority(df_train, LABEL)
print(f"\nBalanced train: {len(df_train_balanced):,} rows | "
      f"SNAP distribution: {dict(Counter(df_train_balanced[LABEL]))}")

### Prompt Builders

In [ ]:
## structured - simple JSON like before
def build_structured_prompt(row, features, instruction=INSTRUCTION_STRUCTURED):
    user_content = "\n".join(
        f"{COL_MAP[k]}: {row[k]}" for k in features if k in row
    )
    return {
        "prompt":     [{"role": "system", "content": instruction},
                       {"role": "user",   "content": user_content}],
        "completion": [{"role": "assistant", "content": str(row[LABEL])}],
    }

def build_narrative_prompt(row, features, instruction=INSTRUCTION_NARRATIVE):
    age         = row.get("age", "unknown age")
    sex         = row.get("sex", "").lower()
    hispanic    = row.get("hispanic.origin", "")
    race        = row.get("race", "")
    ethnicity   = f"{hispanic} {race}".strip()
    marital     = row.get("marital.status", "").lower()
    edu         = row.get("education", "").lower()
    employment  = row.get("employment", "").lower()
    state       = row.get("state", "")
    income      = row.get("income", "")
    citizenship = row.get("citizenship", "").lower()

    if sex == "female":
        person = "woman"
        pronoun = "She"
    elif sex == "male":
        person = "man"
        pronoun = "He"
    else:
        person = "person"
        pronoun = "They"

    ethnicity_parts = []
    if hispanic:
        ethnicity_parts.append(hispanic)
    if race:
        ethnicity_parts.append(race)
    ethnicity = ", ".join(ethnicity_parts)

    first_sentence_parts = [f"This is a {age}-year-old"]
    if ethnicity:
        first_sentence_parts.append(ethnicity)
    first_sentence_parts.append(person)
    first_sentence = " ".join(first_sentence_parts)

    if citizenship:
        first_sentence += f" who is a {citizenship}."
    else:
        first_sentence += "."

    second_sentence = (
        f"{pronoun} {'has' if pronoun == 'She' or pronoun == 'He' else 'have'} "
        f"{marital} and {'holds' if pronoun == 'She' or pronoun == 'He' else 'hold'} "
        f"{'an' if str(edu).lower()[0] in 'aeiou' else 'a'} {edu}."
        if marital and edu else ""
    )

    third_sentence = (
        f"During the current survey month, {pronoun.lower()} was {employment}, "
        f"with a monthly household income of {income}, and resides in {state}."
        if employment and income and state else ""
    )
    narrative = " ".join(
        s for s in [first_sentence, second_sentence, third_sentence] if s
    )

    return {
        "prompt":     [{"role": "system", "content": instruction},
                       {"role": "user",   "content": narrative}],
        "completion": [{"role": "assistant", "content": str(row[LABEL])}],
    }

def make_dataset(df, prompt_fn, features):
    from datasets import Dataset
    records = [prompt_fn(row, features) for _, row in df.iterrows()]
    return Dataset.from_list(records)

### Export to JSON

In [ ]:
def export_to_jsonl(df, prompt_fn, features, path):
    with open(path, "w") as f:
        for _, row in df.iterrows():
            sample   = prompt_fn(row, features)
            messages = sample["prompt"] + sample["completion"]
            f.write(json.dumps({"messages": messages}) + "\n")

experiments_data = {
    "T1A_imbalanced":  (df_train,          build_structured_prompt, FULL_FEATURES,    FULL_FEATURES),
    "T1B_oversampled": (df_train_balanced,  build_structured_prompt, FULL_FEATURES,    FULL_FEATURES),
    "T2A_demo_only":   (df_train,          build_structured_prompt, DEMO_FEATURES,    DEMO_FEATURES),
    "T2C_full_model":  (df_train,          build_structured_prompt, FULL_FEATURES,    FULL_FEATURES),
    "T3B_narrative":   (df_train,          build_narrative_prompt,  FULL_FEATURES,    FULL_FEATURES)
}

os.makedirs("./mlx_data", exist_ok=True)
for run_label, (train_df, prompt_fn, train_feats, val_feats) in experiments_data.items():
    folder = f"./mlx_data/{run_label}"
    os.makedirs(folder, exist_ok=True)
    export_to_jsonl(train_df, prompt_fn, train_feats, f"{folder}/train.jsonl")
    export_to_jsonl(df_test,  prompt_fn, val_feats,   f"{folder}/valid.jsonl")
    print(f"Exported {folder}/")

### Making Test Datasets

In [ ]:
test_ds_structured_full    = make_dataset(df_test, build_structured_prompt, FULL_FEATURES)
test_ds_structured_demo    = make_dataset(df_test, build_structured_prompt, DEMO_FEATURES)
test_ds_narrative_full     = make_dataset(df_test, build_narrative_prompt,  FULL_FEATURES)
print("Test datasets ready.")

### Defining Predictions

In [ ]:
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, classification_report
)

def evaluate_predictions(predictions, references, label_name=""):
    valid         = [(p, r) for p, r in zip(predictions, references) if p in ["Yes", "No"]]
    unknown_count = len(predictions) - len(valid)
    if not valid:
        print("No valid predictions.")
        return {}
    y_pred, y_true = zip(*valid)
    metrics = {
        "Label":     label_name,
        "N_test":    len(df_test),
        "Accuracy":  round(accuracy_score(y_true, y_pred), 4),
        "Recall":    round(recall_score(y_true, y_pred, pos_label="Yes"), 4),
        "Precision": round(precision_score(y_true, y_pred, pos_label="Yes", zero_division=0), 4),
        "F1":        round(f1_score(y_true, y_pred, pos_label="Yes"), 4),
        "Unknown":   unknown_count,
    }
    print(f"\n── {label_name} ──")
    for k, v in metrics.items():
        print(f"  {k}: {v}")
    print(classification_report(y_true, y_pred, zero_division=0))
    return metrics

### Calibration Analytics - ECE computation from Guo et al. 2017

In [ ]:
def calibration_analysis(probabilities, references, label_name):
    y_true_bin = [1 if r == "Yes" else 0 for r in references]
    probs      = np.array(probabilities)

    n_bins    = 10 # this can be changed to however many bins one wants
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece       = 0.0
    for i in range(n_bins):
        mask = (probs >= bin_edges[i]) & (probs < bin_edges[i + 1])
        if mask.sum() == 0:
            continue
        bin_acc  = np.mean(np.array(y_true_bin)[mask])
        bin_conf = np.mean(probs[mask])
        ece     += mask.sum() / len(probs) * abs(bin_acc - bin_conf)
        return round(ece, 4)

### Run Inference Function - one function to compute all metrics together

In [ ]:
from mlx_lm import load, generate

def format_chat_prompt(prompt_list):
    return "".join(
        f"<|{m['role']}|>\n{m['content']}\n" for m in prompt_list
    ) + "<|assistant|>\n"

def run_inference_mlx(model, tokenizer, dataset, return_probs=False):
    predictions, references, probabilities = [], [], []

    for example in tqdm(dataset, desc="Generating"):
        prompt_text = format_chat_prompt(example["prompt"])
        response    = generate(
            model, tokenizer,
            prompt=prompt_text,
            max_tokens=5,
            verbose=False
        )
        match = re.search(r"\b(Yes|No)\b", response, re.IGNORECASE)
        predictions.append(match.group(1).title() if match else "Unknown")
        references.append(example["completion"][0]["content"].strip().title())
        # crude way of mapping 1/0.5/0 as probabilities looking at the returned label
        if return_probs:
            if match:
                probabilities.append(1.0 if match.group(1).title() == "Yes" else 0.0)
            else:
                probabilities.append(0.5)

    if return_probs:
        return predictions, references, probabilities
    return predictions, references

### Lora Configuration

In [ ]:
import yaml

lora_config = {
    "lora_parameters": {
        "rank": 16,
        "scale": 2.0,   # equivalent to alpha=32 with rank=16 (scale = alpha/rank = 32/16)
        "dropout": 0.05,
    }
}

os.makedirs("./configs", exist_ok=True)
with open("./configs/lora_config.yaml", "w") as f:
    yaml.dump(lora_config, f)

print("LoRA config saved to ./configs/lora_config.yaml")

### Training and Inference for T1A_imbalanced

In [ ]:
!mlx_lm.lora \
  --model mlx-community/Qwen2.5-1.5B-Instruct-4bit \
  --train \
  --data ./mlx_data/T1A_imbalanced \
  --num-layers 16 \
  --batch-size 4 \
  --grad-accumulation-steps 4 \
  --iters 2000 \
  --learning-rate 2e-4 \
  --mask-prompt \
  --adapter-path ./adapters_w1/T1A_imbalanced \
  --save-every 500 \
  -c ./configs/lora_config.yaml

In [ ]:
import time

all_metrics = []
experiment_times = {}

# Load model with T1-A adapter
model_mlx, tok_mlx = load(
    "mlx-community/Qwen2.5-1.5B-Instruct-4bit",
    adapter_path="./adapters_w1/T1A_imbalanced",
)

#  Inference
t0 = time.time()
preds, refs, probs = run_inference_mlx(
    model_mlx, tok_mlx, test_ds_structured_full, return_probs=True
)

#  Evaluate
metrics = evaluate_predictions(preds, refs, label_name="T1A_imbalanced")

#  Calibration
ece = calibration_analysis(
    probs, refs,
    label_name="T1A_imbalanced")
metrics["ECE"] = ece
all_metrics.append(metrics)

# Save predictions
out_df = pd.DataFrame({"Predicted_Label": preds, "Ground_Truth": refs, "P_Yes": probs})
out_df.to_csv(os.path.join(OUTPUT_DIR, "predictions_T1A_imbalanced.csv"), index=False)

elapsed = round(time.time() - t0, 2)
experiment_times["T1A_imbalanced"] = elapsed
print(f"\n  T1A_imbalanced done in {elapsed:.1f}s")

# Clean up
del model_mlx, tok_mlx

### Training and Inference for T1B_oversampled

In [ ]:
!mlx_lm.lora \
  --model mlx-community/Qwen2.5-1.5B-Instruct-4bit \
  --train \
  --data ./mlx_data/T1B_oversampled \
  --num-layers 16 \
  --batch-size 4 \
  --grad-accumulation-steps 4 \
  --iters 2000 \
  --learning-rate 2e-4 \
  --mask-prompt \
  --adapter-path ./adapters_w1/T1B_oversampled \
  --save-every 500 \
  -c ./configs/lora_config.yaml

In [ ]:
t0 = time.time()

model_mlx, tok_mlx = load(
    "mlx-community/Qwen2.5-1.5B-Instruct-4bit",
    adapter_path="./adapters_w1/T1B_oversampled",
)

preds, refs, probs = run_inference_mlx(
    model_mlx, tok_mlx, test_ds_structured_full, return_probs=True
)

metrics = evaluate_predictions(preds, refs, label_name="T1B_oversampled")

ece = calibration_analysis(
    probs, refs,
    label_name="T1B_oversampled")
metrics["ECE"] = ece
all_metrics.append(metrics)

out_df = pd.DataFrame({"Predicted_Label": preds, "Ground_Truth": refs, "P_Yes": probs})
out_df.to_csv(os.path.join(OUTPUT_DIR, "predictions_T1B_oversampled.csv"), index=False)

elapsed = round(time.time() - t0, 2)
experiment_times["T1B_oversampled"] = elapsed
print(f"\n  T1B_oversampled done in {elapsed:.1f}s")

del model_mlx, tok_mlx

### Training and Inference for T2A_demo_only

In [ ]:
!mlx_lm.lora \
  --model mlx-community/Qwen2.5-1.5B-Instruct-4bit \
  --train \
  --data ./mlx_data/T2A_demo_only \
  --num-layers 16 \
  --batch-size 4 \
  --grad-accumulation-steps 4 \
  --iters 2000 \
  --learning-rate 2e-4 \
  --mask-prompt \
  --adapter-path ./adapters_w1/T2A_demo_only \
  --save-every 500 \
  -c ./configs/lora_config.yaml

In [ ]:
t0 = time.time()

model_mlx, tok_mlx = load(
    "mlx-community/Qwen2.5-1.5B-Instruct-4bit",
    adapter_path="./adapters_w1/T2A_demo_only",
)

preds, refs, probs = run_inference_mlx(
    model_mlx, tok_mlx, test_ds_structured_demo, return_probs=True
)

metrics = evaluate_predictions(preds, refs, label_name="T2A_demo_only")

ece = calibration_analysis(
    probs, refs,
    label_name="T2A_demo_only"
)
metrics["ECE"] = ece
all_metrics.append(metrics)

out_df = pd.DataFrame({"Predicted_Label": preds, "Ground_Truth": refs, "P_Yes": probs})
out_df.to_csv(os.path.join(OUTPUT_DIR, "predictions_T2A_demo_only.csv"), index=False)

elapsed = round(time.time() - t0, 2)
experiment_times["T2A_demo_only"] = elapsed
print(f"\n  T2A_demo_only done in {elapsed:.1f}s")

del model_mlx, tok_mlx

### Training and Inference for T2C_full_model

In [ ]:
!mlx_lm.lora \
  --model mlx-community/Qwen2.5-1.5B-Instruct-4bit \
  --train \
  --data ./mlx_data/T2C_full_model \
  --num-layers 16 \
  --batch-size 4 \
  --grad-accumulation-steps 4 \
  --iters 2000 \
  --learning-rate 2e-4 \
  --mask-prompt \
  --adapter-path ./adapters_w1/T2C_full_model \
  --save-every 500 \
  -c ./configs/lora_config.yaml

In [ ]:
t0 = time.time()

model_mlx, tok_mlx = load(
    "mlx-community/Qwen2.5-1.5B-Instruct-4bit",
    adapter_path="./adapters_w1/T2C_full_model",
)

preds, refs, probs = run_inference_mlx(
    model_mlx, tok_mlx, test_ds_structured_full, return_probs=True
)

metrics = evaluate_predictions(preds, refs, label_name="T2C_full_model")

ece = calibration_analysis(
    probs, refs,
    label_name="T2C_full_model"
)
metrics["ECE"] = ece
all_metrics.append(metrics)

out_df = pd.DataFrame({"Predicted_Label": preds, "Ground_Truth": refs, "P_Yes": probs})
out_df.to_csv(os.path.join(OUTPUT_DIR, "predictions_T2C_full_model.csv"), index=False)

elapsed = round(time.time() - t0, 2)
experiment_times["T2C_full_model"] = elapsed
print(f"\n  T2C_full_model done in {elapsed:.1f}s")

del model_mlx, tok_mlx

### Inference for Zero-shot model

In [ ]:
t0 = time.time()

model_mlx, tok_mlx = load(
    "mlx-community/Qwen2.5-1.5B-Instruct-4bit",
    # no adapter_path — base model only
)

preds, refs, probs = run_inference_mlx(
    model_mlx, tok_mlx, test_ds_structured_full, return_probs=True
)

metrics = evaluate_predictions(preds, refs, label_name="T3A_zeroshot")

ece = calibration_analysis(
    probs, refs,
    label_name="T3A_zeroshot")
metrics["ECE"] = ece
all_metrics.append(metrics)

out_df = pd.DataFrame({"Predicted_Label": preds, "Ground_Truth": refs, "P_Yes": probs})
out_df.to_csv(os.path.join(OUTPUT_DIR, "predictions_T3A_zeroshot.csv"), index=False)

elapsed = round(time.time() - t0, 2)
experiment_times["T3A_zeroshot"] = elapsed
print(f"\n  T3A_zeroshot done in {elapsed:.1f}s")

del model_mlx, tok_mlx

### Training and Inference for T3B_narrative

In [ ]:
!mlx_lm.lora \
  --model mlx-community/Qwen2.5-1.5B-Instruct-4bit \
  --train \
  --data ./mlx_data/T3B_narrative \
  --num-layers 16 \
  --batch-size 4 \
  --grad-accumulation-steps 4 \
  --iters 2000 \
  --learning-rate 2e-4 \
  --mask-prompt \
  --adapter-path ./adapters_w1/T3B_narrative \
  --save-every 500 \
  -c ./configs/lora_config.yaml

In [ ]:
t0 = time.time()

model_mlx, tok_mlx = load(
    "mlx-community/Qwen2.5-1.5B-Instruct-4bit",
    adapter_path="./adapters_w1/T3B_narrative",
)

preds, refs, probs = run_inference_mlx(
    model_mlx, tok_mlx, test_ds_narrative_full, return_probs=True
)

metrics = evaluate_predictions(preds, refs, label_name="T3B_narrative")

ece = calibration_analysis(
    probs, refs,
    label_name="T3B_narrative"
)
metrics["ECE"] = ece
all_metrics.append(metrics)

out_df = pd.DataFrame({"Predicted_Label": preds, "Ground_Truth": refs, "P_Yes": probs})
out_df.to_csv(os.path.join(OUTPUT_DIR, "predictions_T3B_narrative.csv"), index=False)

elapsed = round(time.time() - t0, 2)
experiment_times["T3B_narrative"] = elapsed
print(f"\n  T3B_narrative done in {elapsed:.1f}s")

del model_mlx, tok_mlx

In [ ]:
print(f"Experiments collected: {len(all_metrics)}")
for m in all_metrics:
    print(f"  {m['Label']} — F1: {m['F1']}")

### Summary Table

In [ ]:
import pandas as pd
import os

#  Build summary dataframe from accumulated metrics
summary_df = pd.DataFrame(all_metrics)

# Reorder columns cleanly
col_order = ["Label", "N_test", "Accuracy", "Recall", "Precision", "F1", "ECE", "Unknown"]
summary_df = summary_df[[c for c in col_order if c in summary_df.columns]]

#  Save to CSV
summary_path = os.path.join(OUTPUT_DIR, "llm_wave1_summary.csv")
summary_df.to_csv(summary_path, index=False)
print(f"Summary saved → {summary_path}\n")

# Save runtimes
runtime_df = pd.DataFrame(
    list(experiment_times.items()), columns=["Experiment", "Runtime_seconds"]
)
runtime_df["Runtime_minutes"] = (runtime_df["Runtime_seconds"] / 60).round(2)
runtime_path = os.path.join(OUTPUT_DIR, "runtimes_wave1.csv")
runtime_df.to_csv(runtime_path, index=False)
print(f"Runtimes saved → {runtime_path}\n")